# M24 lab solution: Agentic Workflow Patterns

**Brief.** Pattern mapping: rebuild query decomposition (M17) as orchestrator–worker and an answer-quality check as evaluator–optimizer, then compare latency and cost with the chain versions.

Run with the default environment: `uv run jupyter lab`.

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # API keys from the .env file at the repo root
sys.path.insert(0, str(Path("../../../../src").resolve()))

from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k  # noqa: E402

DATA = Path("../data/larkspur")
handbook = load_handbook(DATA / "handbook")
golden = load_jsonl(DATA / "golden.jsonl")
by_id = {g["id"]: g for g in golden}
scored = [g for g in golden if g["evidence_spans"]]
print(len(handbook), "handbook files,", len(golden), "golden questions,", len(scored), "with evidence")

In [ ]:
from helpers import get_embeddings, get_experientiallabs_llm

llm = get_experientiallabs_llm()
embeddings = get_embeddings()
LLM_NAME = llm.model_name
EMBEDDING_NAME = getattr(embeddings, "model", None) or getattr(embeddings, "endpoint", None)
print("LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)

In [ ]:
# The inline "Part V system", reused by the later labs: the handbook split as lab_kit.baseline splits it
# (recursive, 500 characters, 50 overlap), a FAISS vector retriever over those chunks, and an answer
# prompt that cites numbered passages and is allowed to say "I don't know".
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [
    Document(page_content=c, metadata={"source": p["source"], "section": p["title"]})
    for p in handbook
    for c in splitter.split_text(p["text"])
]
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions from Larkspur Cloud customers using only the numbered handbook passages. "
               "Cite the passages you use as [1], [2]. If the passages do not contain the answer, "
               "reply exactly: I don't know."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata.get('source', '?')}) {d.page_content}" for i, d in enumerate(docs, 1))


def answer(question, docs, config=None):
    return (ANSWER_PROMPT | llm).invoke({"context": format_docs(docs), "question": question}, config=config).content


print(len(chunks), "chunks indexed")

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler

# Prices in USD per million tokens. Fill them from the provider's price page on the day you run the
# lab and note that date in the README; with None the cost column stays empty.
PRICE_IN_PER_1M = None
PRICE_OUT_PER_1M = None


class Meter(BaseCallbackHandler):
    """Counts chat-model calls and tokens for everything run with config={"callbacks": [meter]}."""

    def __init__(self):
        self.calls = self.tokens_in = self.tokens_out = 0

    def on_llm_end(self, response, **kwargs):
        for gens in response.generations:
            for gen in gens:
                self.calls += 1
                usage = getattr(getattr(gen, "message", None), "usage_metadata", None) or {}
                self.tokens_in += usage.get("input_tokens", 0)
                self.tokens_out += usage.get("output_tokens", 0)

    def cost(self):
        if PRICE_IN_PER_1M is None or PRICE_OUT_PER_1M is None:
            return None
        return (self.tokens_in * PRICE_IN_PER_1M + self.tokens_out * PRICE_OUT_PER_1M) / 1e6

In [ ]:
QUESTIONS = [by_id[q] for q in ["g02", "g17", "g18", "g19", "g20", "g21", "g01", "g03", "g09", "g12"]]
print([(q["id"], q["type"]) for q in QUESTIONS])

## 1. Query decomposition: chain, then orchestrator and workers

The chain version is the M17 decomposition as one LCEL sequence: split the question, retrieve 2 per sub-question, answer from all passages at once. The orchestrator-worker version is a LangGraph graph: the orchestrator writes the sub-questions, `Send` starts one worker per sub-question in parallel (each retrieves and answers its part), and a synthesiser writes the final answer from the partial answers.

In [ ]:
import operator
from typing import Annotated, TypedDict

from langchain_core.runnables import RunnableLambda
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel, Field


class SubQuestions(BaseModel):
    questions: list[str] = Field(description="At most three self-contained sub-questions, one per fact needed.")


decompose = ChatPromptTemplate.from_messages([
    ("system", "Split the question into at most three self-contained sub-questions, one per fact it needs. "
               "If it needs one fact, return it unchanged as the only item."),
    ("human", "{question}"),
]) | llm.with_structured_output(SubQuestions)


def gather(inputs):
    subs = inputs["subs"].questions[:3]
    docs = [d for s in subs for d in vectorstore.similarity_search(s, k=2)]
    return {"question": inputs["question"], "context": format_docs(docs)}


decomposition_chain = (
    {"question": lambda x: x["question"], "subs": decompose}
    | RunnableLambda(gather)
    | ANSWER_PROMPT
    | llm
)


class OWState(TypedDict):
    question: str
    subs: list[str]
    partials: Annotated[list[str], operator.add]
    answer: str


class WorkerState(TypedDict):
    sub: str


def orchestrator(state, config):
    return {"subs": decompose.invoke({"question": state["question"]}, config=config).questions[:3]}


def fan_out(state):
    return [Send("worker", {"sub": s}) for s in state["subs"]]


def worker(state, config):
    docs = vectorstore.similarity_search(state["sub"], k=2)
    return {"partials": [f"Sub-question: {state['sub']}\nAnswer: {answer(state['sub'], docs, config=config)}"]}


SYNTH = ChatPromptTemplate.from_messages([
    ("system", "Combine the partial answers into one answer to the customer's question. Keep their citations. "
               "If they do not answer it, reply exactly: I don't know."),
    ("human", "Question: {question}\n\n{partials}"),
])


def synthesise(state, config):
    reply = (SYNTH | llm).invoke({"question": state["question"], "partials": "\n\n".join(state["partials"])}, config=config)
    return {"answer": reply.content}


g = StateGraph(OWState)
g.add_node("orchestrator", orchestrator)
g.add_node("worker", worker)
g.add_node("synthesise", synthesise)
g.add_edge(START, "orchestrator")
g.add_conditional_edges("orchestrator", fan_out, ["worker"])
g.add_edge("worker", "synthesise")
g.add_edge("synthesise", END)
orchestrator_worker = g.compile()
print(orchestrator_worker.get_graph().draw_mermaid())

## 2. Answer-quality check: chain, then evaluator and optimizer

The chain version answers, checks the answer once against the passages, and rewrites it once if the check fails. The evaluator-optimizer version loops: generate, evaluate, and regenerate with the evaluator's feedback until the answer passes or `MAX_ITERATIONS` is reached. The counter is bumped inside the generator node, because a router's state writes are discarded.

In [ ]:
class Check(BaseModel):
    grounded: bool = Field(description="True when every claim in the answer is supported by the passages and it answers the question.")
    feedback: str = Field(description="What to fix, or an empty string when grounded.")


evaluate = ChatPromptTemplate.from_messages([
    ("system", "Check an answer against the handbook passages it was written from."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}\n\nAnswer: {answer}"),
]) | llm.with_structured_output(Check)
REVISE = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the answer so it uses only the numbered passages and fixes the feedback. Cite passages as [1], [2]. "
               "If the passages do not contain the answer, reply exactly: I don't know."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}\n\nPrevious answer: {answer}\n\nFeedback: {feedback}"),
])
MAX_ITERATIONS = 3


def check_chain(question, config):
    context = format_docs(retriever.invoke(question))
    draft = (ANSWER_PROMPT | llm).invoke({"context": context, "question": question}, config=config).content
    verdict = evaluate.invoke({"context": context, "question": question, "answer": draft}, config=config)
    if verdict.grounded:
        return draft, 1
    revised = (REVISE | llm).invoke({"context": context, "question": question, "answer": draft,
                                     "feedback": verdict.feedback}, config=config).content
    return revised, 2


class EOState(TypedDict):
    question: str
    context: str
    answer: str
    feedback: str
    grounded: bool
    iterations: int


def generate(state, config):
    if state.get("answer"):
        reply = (REVISE | llm).invoke({"context": state["context"], "question": state["question"],
                                       "answer": state["answer"], "feedback": state["feedback"]}, config=config)
    else:
        reply = (ANSWER_PROMPT | llm).invoke({"context": state["context"], "question": state["question"]}, config=config)
    return {"answer": reply.content, "iterations": state.get("iterations", 0) + 1}


def evaluator(state, config):
    v = evaluate.invoke({"context": state["context"], "question": state["question"], "answer": state["answer"]}, config=config)
    return {"grounded": v.grounded, "feedback": v.feedback}


def route(state):
    return END if state["grounded"] or state["iterations"] >= MAX_ITERATIONS else "generate"


e = StateGraph(EOState)
e.add_node("generate", generate)
e.add_node("evaluate", evaluator)
e.add_edge(START, "generate")
e.add_edge("generate", "evaluate")
e.add_conditional_edges("evaluate", route, ["generate", END])
evaluator_optimizer = e.compile()

## 3. Latency and cost on the 10 questions

In [ ]:
rows = []


def run_loop(question, config):
    out = evaluator_optimizer.invoke({"question": question, "context": format_docs(retriever.invoke(question))},
                                     config={**config, "recursion_limit": 2 * MAX_ITERATIONS + 2})
    return out["answer"], out["iterations"]


def measure(pattern, version, q, fn):
    meter = Meter()
    t0 = time.perf_counter()
    reply, iterations = fn({"callbacks": [meter]})
    rows.append({"pattern": pattern, "version": version, "id": q["id"], "type": q["type"],
                 "seconds": round(time.perf_counter() - t0, 2), "llm_calls": meter.calls,
                 "tokens": meter.tokens_in + meter.tokens_out, "usd": meter.cost(), "iterations": iterations,
                 "answer": reply})


for q in QUESTIONS:
    text = q["question"]
    measure("decomposition", "chain", q, lambda cfg: (decomposition_chain.invoke({"question": text}, config=cfg).content, None))
    measure("decomposition", "orchestrator-worker", q,
            lambda cfg: (orchestrator_worker.invoke({"question": text}, config=cfg)["answer"], None))
    measure("quality check", "chain", q, lambda cfg: check_chain(text, cfg))
    measure("quality check", "evaluator-optimizer", q, lambda cfg: run_loop(text, cfg))
    print(q["id"], "done")

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
print("run:", time.strftime("%Y-%m-%d"), "| LLM:", LLM_NAME, "| embeddings:", EMBEDDING_NAME)
print(df.groupby(["pattern", "version"])[["seconds", "llm_calls", "tokens", "usd", "iterations"]].mean().round(3).to_string())
print()
print(df.groupby(["pattern", "version", "type"])[["seconds", "llm_calls"]].mean().round(2).to_string())

## What this shows

- Orchestrator-worker spends more LLM calls than the chain (one answer per sub-question plus a synthesis), and wins back time only when the workers run in parallel and there is more than one of them; the lookup rows, which usually decompose into one sub-question, show the overhead alone.
- The evaluator-optimizer loop costs one generate and one evaluate per iteration. Its mean iterations column says how often the first answer already passed, which is how often the loop was pure overhead.
- Both patterns hand the model one decision (how many sub-questions; pass or retry) and keep the path in code, with a hard cap on the loop.